# Exercise 2.1: 50-Member Ensemble Forecast

**Level**: Intermediate | **Time**: 30 min | **GPU**: Required (T4 works!)

Generate a probabilistic weather forecast with 50 ensemble members. Learn why ensembles matter more than single deterministic forecasts.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

---

**Why ensembles matter:**
- A deterministic forecast says "it will be 30°C"
- An ensemble says "70% chance of 28-32°C, but 15% chance of >35°C"
- The second is far more useful for energy grids, emergency managers, agriculture
- FCN3 generates 50 members in the time traditional NWP takes for ONE

In [ ]:
# Install Earth2Studio + FCN3 dependencies (don't install torch — Colab has it)
# makani is required by FCN3 but not on PyPI, must install from GitHub
!pip install -q "earth2studio>=0.13.0" torch-harmonics matplotlib xarray zarr scipy
!pip install -q "makani @ git+https://github.com/NVIDIA/makani.git"

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. This notebook needs a GPU.")
    print("In Colab: Runtime → Change runtime type → T4 GPU")

## Part 1: Run the Ensemble

In [ ]:
from earth2studio.models.px import FCN3
from earth2studio.data import GFS
from earth2studio.io import ZarrBackend
from earth2studio.perturbation import SphericalGaussian
from earth2studio import run

# Load model
print("Loading FCN3...")
model = FCN3.load_model(FCN3.load_default_package())

# SphericalGaussian adds perturbations that respect Earth's spherical geometry
# noise_amplitude=0.05 is a good starting point
perturbation = SphericalGaussian(noise_amplitude=0.05)

print("Running 50-member ensemble (10-day forecast)...")
io = run.ensemble(
    time=["2025-06-01T00:00:00"],
    nsteps=40,        # 10 days (40 × 6hr)
    nensemble=50,      # 50 members
    model=model,
    data=GFS(),
    io=ZarrBackend("ensemble.zarr"),
    perturbation=perturbation
)
print("Done!")

## Part 2: Ensemble Statistics

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

ds = xr.open_zarr("ensemble.zarr")
print(f"Shape: {dict(ds.dims)}")
# Expected: (time=1, ensemble=50, lead_time=41, variable=73, lat=721, lon=1440)

lats = ds.coords['lat'].values
lons = ds.coords['lon'].values

# Get t2m at T+120h (5 days)
step = 20
t2m = ds.sel(variable="t2m").isel(time=0, lead_time=step) - 273.15  # to °C

ens_mean = t2m.mean(dim="ensemble").values.squeeze()
ens_std = t2m.std(dim="ensemble").values.squeeze()

print(f"\nT+{step*6}h ensemble stats:")
print(f"  Global mean temp: {np.nanmean(ens_mean):.1f}°C")
print(f"  Mean spread (σ):  {np.nanmean(ens_std):.2f}°C")
print(f"  Max spread:       {np.nanmax(ens_std):.2f}°C")

## Part 3: Ensemble Mean + Spread Map

**High spread = high uncertainty.** Look at where the ensemble members disagree — these are the regions where the forecast is least certain.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Ensemble mean
cf1 = axes[0].contourf(lons, lats, ens_mean, levels=np.arange(-40, 45, 5), cmap='RdYlBu_r')
axes[0].set_title(f'Ensemble Mean T2m (°C) — T+{step*6}h')
plt.colorbar(cf1, ax=axes[0], label='Temperature (°C)')

# Ensemble spread
cf2 = axes[1].contourf(lons, lats, ens_std, levels=np.arange(0, 8, 0.5), cmap='YlOrRd')
axes[1].set_title(f'Ensemble Spread (σ) — T+{step*6}h\nHigh = Uncertain')
plt.colorbar(cf2, ax=axes[1], label='Std Dev (°C)')

for ax in axes:
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

fig.suptitle('50-Member FCN3 Ensemble', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Part 4: Spread Growth Over Time

In a chaotic system, ensemble members diverge exponentially. This curve shows how quickly uncertainty grows.

In [ ]:
t2m_all = ds.sel(variable="t2m").isel(time=0) - 273.15

spreads = []
lead_hours = []
for s in range(len(ds.coords['lead_time'])):
    std = t2m_all.isel(lead_time=s).std(dim="ensemble").values.squeeze()
    spreads.append(np.nanmean(std))
    lead_hours.append(s * 6)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(lead_hours, spreads, 'r-o', markersize=3, linewidth=2)
ax.set_xlabel('Forecast Lead Time (hours)')
ax.set_ylabel('Mean Global Ensemble Spread (σ °C)')
ax.set_title('Ensemble Spread Growth — 50-member FCN3')
ax.grid(True, alpha=0.3)
for d, label in [(72, 'Day 3'), (120, 'Day 5'), (240, 'Day 10')]:
    ax.axvline(x=d, color='gray', linestyle='--', alpha=0.4)
    ax.text(d+2, max(spreads)*0.95, label, fontsize=9, color='gray')
plt.tight_layout()
plt.show()

## Part 5: Probability of Extreme Heat

**This is the real power of ensembles** — probabilistic forecasting.

"What's the probability of temperature exceeding 35°C (95°F) in 5 days?"

In [ ]:
threshold = 35.0  # °C
t2m_step = ds.sel(variable="t2m").isel(time=0, lead_time=20) - 273.15

# Count members exceeding threshold at each grid point
exceed = (t2m_step > threshold).sum(dim="ensemble").values.squeeze()
prob = exceed / len(ds.coords['ensemble']) * 100

fig, ax = plt.subplots(figsize=(14, 6))
cf = ax.contourf(lons, lats, prob,
                 levels=[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
                 cmap='YlOrRd')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'Probability of T2m > {threshold}°C — T+120h\n'
             f'50-member FCN3 Ensemble', fontsize=14)
plt.colorbar(cf, ax=ax, label='Probability (%)')
plt.tight_layout()
plt.show()

print(f"Max probability: {np.nanmax(prob):.0f}%")
print(f"Grid points with P > 50%: {np.sum(prob > 50)}")

## Part 6: Spaghetti Plot for a Single Location

All 50 ensemble members plotted for one city. The "cone of uncertainty" — members diverge as forecast extends further.

In [ ]:
# Pick a city (lat, lon in 0-360 convention)
cities = {
    "New York": (40.7, 286.0),
    "London": (51.5, 359.9),
    "Tokyo": (35.7, 139.7),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, (target_lat, target_lon)) in zip(axes, cities.items()):
    lat_idx = np.argmin(np.abs(lats - target_lat))
    lon_idx = np.argmin(np.abs(lons - target_lon))
    
    # Extract all members for this point
    t2m_point = ds.sel(variable="t2m").isel(time=0).values[:, :, lat_idx, lon_idx] - 273.15
    # Shape: (ensemble, lead_time)
    
    hours = np.arange(t2m_point.shape[1]) * 6
    
    # Plot each member
    for m in range(t2m_point.shape[0]):
        ax.plot(hours, t2m_point[m], color='steelblue', alpha=0.15, linewidth=0.8)
    
    # Mean and percentiles
    mean = np.mean(t2m_point, axis=0)
    p10 = np.percentile(t2m_point, 10, axis=0)
    p90 = np.percentile(t2m_point, 90, axis=0)
    
    ax.plot(hours, mean, 'r-', linewidth=2.5, label='Mean')
    ax.fill_between(hours, p10, p90, color='red', alpha=0.15, label='10-90th %ile')
    
    ax.set_xlabel('Lead Time (hours)')
    ax.set_ylabel('Temperature (°C)')
    ax.set_title(name)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Spaghetti Plots — 50-Member FCN3 Ensemble', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Exercises

1. **Change noise_amplitude** to 0.01 and 0.20. How does spread change?
2. **Where is spread largest?** (Hint: midlatitude storm tracks, tropical convection)
3. **Frost probability**: Create a map of P(T2m < 0°C) at T+72h
4. **Different cities**: Add your city to the spaghetti plot
5. **CRPS**: Compute the Continuous Ranked Probability Score against ERA5

**Next**: Exercise 2.2 (CorrDiff Downscaling) and 2.3 (Custom Diagnostics)